# UCI SECOM Baseline Reproduction Attempt

This notebook attempts to reproduce the feature-selection baseline described in the [UCI SECOM dataset notes](https://archive.ics.uci.edu/dataset/179/secom).

The reproduction is **approximate** because the UCI summary reports the feature-selection methods, number of selected features, classifier family, and 10-fold cross-validation, but it does not provide every implementation detail or hyperparameter value.

## Contents

1. [Goal and Assumptions](#1-goal-and-assumptions)
2. [Data Loading](#2-data-loading)
3. [Kernel Ridge Classifier Wrapper](#3-kernel-ridge-classifier-wrapper)
4. [F-Test Feature Selection Baseline](#4-f-test-feature-selection-baseline)
5. [Optional Pearson Feature Ranking](#5-optional-pearson-feature-ranking)
6. [Comparison with UCI Published Baseline](#6-comparison-with-uci-published-baseline)
7. [Discussion](#7-discussion)
8. [References](#8-references)

## 1. Goal and Assumptions

The UCI notes report a simple baseline using:

- standardized data;
- constant-feature removal;
- feature selection with the top `40` ranked features;
- a simple kernel ridge classifier;
- 10-fold cross-validation;
- balanced error rate (`BER`) as the main metric.

Assumptions used here:

- parameters above `50%` missingness are removed using training-fold data;
- remaining missing values are imputed with the training-fold median;
- constant parameters are removed inside each fold;
- F-test ranking is implemented with `SelectKBest(f_classif, k=40)`;
- Kernel Ridge uses sklearn's `KernelRidge` through a small classifier wrapper;
- an RBF kernel with `alpha=1.0` and default `gamma` is used as a transparent fixed configuration;
- no hyperparameters are tuned on the complete dataset.

The source for this benchmark description is the official [UCI SECOM dataset page](https://archive.ics.uci.edu/dataset/179/secom), which reports preprocessing, 40-feature selection, a simple kernel ridge classifier, 10-fold cross-validation, and BER values.


In [1]:
# Imports
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, r_regression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, recall_score, balanced_accuracy_score

from src.data_loading import load_secom_data
from src.modeling import KernelRidgeClassifier
from src.preprocessing import DEFAULT_MAX_MISSING_FRACTION, MissingValueColumnFilter

## 2. Data Loading

The raw SECOM files are loaded from `data/raw`. The label convention is the same as in the main notebook:

- `1` = Fail / faulty;
- `-1` = Pass / normal.

In [2]:
DATA_DIR = PROJECT_ROOT / "data" / "raw"

X, y = load_secom_data(DATA_DIR)

print(f"Feature matrix shape: {X.shape}")
print("Target distribution:")
print(y.value_counts().sort_index())


Feature matrix shape: (1567, 590)
Target distribution:
target
-1    1463
 1     104
Name: count, dtype: int64


## 3. Kernel Ridge Classifier Wrapper

`sklearn.kernel_ridge.KernelRidge` is a regression estimator. The project wrapper in `src/modeling.py` makes it usable as a classifier by:

- fitting on labels `-1` and `1`;
- using the continuous regression output as a decision score;
- predicting Fail when the score is at least `0`;
- exposing `classes_ = np.array([-1, 1])` for sklearn compatibility.

In [3]:
kernel_ridge_classifier = KernelRidgeClassifier(
    alpha=1.0,
    kernel="rbf",
    gamma=None,
)

print(kernel_ridge_classifier)


KernelRidgeClassifier()


## 4. F-Test Feature Selection Baseline

This is the primary reproduction attempt. All preprocessing and feature selection steps are inside the pipeline, so each fold learns its own high-missingness filter, imputer, constant-feature filter, scaler, and selected features from the training partition only.

In [4]:
f_test_pipeline = Pipeline(steps=[
    ("missingness_filter", MissingValueColumnFilter(DEFAULT_MAX_MISSING_FRACTION)),
    ("imputer", SimpleImputer(strategy="median")),
    ("variance_filter", VarianceThreshold(threshold=0.0)),
    ("scaler", StandardScaler()),
    ("feature_selector", SelectKBest(score_func=f_classif, k=40)),
    ("classifier", KernelRidgeClassifier(alpha=1.0, kernel="rbf", gamma=None)),
])

cv_10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

scoring = {
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
    "fail_recall": make_scorer(recall_score, pos_label=1, zero_division=0),
    "pass_recall": make_scorer(recall_score, pos_label=-1, zero_division=0),
}

f_test_cv_results = cross_validate(
    f_test_pipeline,
    X,
    y,
    cv=cv_10,
    scoring=scoring,
    n_jobs=1,
    return_train_score=False,
)

f_test_cv_results.keys()

dict_keys(['fit_time', 'score_time', 'test_balanced_accuracy', 'test_fail_recall', 'test_pass_recall'])

In [5]:
def summarize_uci_cv_results(model_name, cv_results):
    rows = []
    balanced_accuracy = cv_results["test_balanced_accuracy"]
    ber = 1 - balanced_accuracy

    metric_values = {
        "balanced_accuracy": balanced_accuracy,
        "balanced_error_rate": ber,
        "fail_recall": cv_results["test_fail_recall"],
        "pass_recall": cv_results["test_pass_recall"],
    }

    for metric, values in metric_values.items():
        rows.append({
            "model": model_name,
            "metric": metric,
            "mean": np.mean(values),
            "std": np.std(values, ddof=1),
        })

    return pd.DataFrame(rows)

f_test_summary = summarize_uci_cv_results(
    "F-test + KernelRidge",
    f_test_cv_results,
)

f_test_summary.pivot(index="model", columns="metric", values="mean").round(3)

metric,balanced_accuracy,balanced_error_rate,fail_recall,pass_recall
model,,,,
F-test + KernelRidge,0.506,0.494,0.02,0.992


In [6]:
f_test_summary_display = f_test_summary.copy()
f_test_summary_display["mean_percent"] = f_test_summary_display["mean"] * 100
f_test_summary_display["std_percent"] = f_test_summary_display["std"] * 100

f_test_summary_display[["model", "metric", "mean_percent", "std_percent"]].round(2)

,model,metric,mean_percent,std_percent
0,F-test + KernelRidge,balanced_accuracy,50.59,2.08
1,F-test + KernelRidge,balanced_error_rate,49.41,2.08
2,F-test + KernelRidge,fail_recall,2.00,4.22
3,F-test + KernelRidge,pass_recall,99.18,0.71


## 5. Optional Pearson Feature Ranking

The UCI notes also report a Pearson feature-ranking baseline. This optional comparison uses absolute Pearson correlation scores through `r_regression` and still keeps feature selection inside the cross-validation pipeline.

In [7]:
def pearson_abs_score(X_fold, y_fold):
    scores = np.abs(r_regression(X_fold, y_fold, force_finite=True))
    p_values = np.full_like(scores, fill_value=np.nan, dtype=float)
    return scores, p_values

pearson_pipeline = Pipeline(steps=[
    ("missingness_filter", MissingValueColumnFilter(DEFAULT_MAX_MISSING_FRACTION)),
    ("imputer", SimpleImputer(strategy="median")),
    ("variance_filter", VarianceThreshold(threshold=0.0)),
    ("scaler", StandardScaler()),
    ("feature_selector", SelectKBest(score_func=pearson_abs_score, k=40)),
    ("classifier", KernelRidgeClassifier(alpha=1.0, kernel="rbf", gamma=None)),
])

pearson_cv_results = cross_validate(
    pearson_pipeline,
    X,
    y,
    cv=cv_10,
    scoring=scoring,
    n_jobs=1,
    return_train_score=False,
)

pearson_summary = summarize_uci_cv_results(
    "Pearson + KernelRidge",
    pearson_cv_results,
)

pd.concat([f_test_summary, pearson_summary], ignore_index=True).pivot(
    index="model",
    columns="metric",
    values="mean",
).round(3)

metric,balanced_accuracy,balanced_error_rate,fail_recall,pass_recall
model,,,,
F-test + KernelRidge,0.506,0.494,0.02,0.992
Pearson + KernelRidge,0.506,0.494,0.02,0.992


## 6. Comparison with UCI Published Baseline

The table below compares this approximate reproduction with the values reported in the [UCI SECOM notes](https://archive.ics.uci.edu/dataset/179/secom). UCI reports values as percentages.

In [8]:
uci_published = pd.DataFrame([
    {
        "method": "UCI F-test baseline",
        "ber_mean_percent": 33.5,
        "ber_std_percent": 2.2,
        "fail_recall_mean_percent": 59.1,
        "fail_recall_std_percent": 4.8,
        "pass_recall_mean_percent": 73.8,
        "pass_recall_std_percent": 1.8,
    },
    {
        "method": "UCI Pearson baseline",
        "ber_mean_percent": 34.1,
        "ber_std_percent": 2.0,
        "fail_recall_mean_percent": 57.4,
        "fail_recall_std_percent": 4.3,
        "pass_recall_mean_percent": 74.4,
        "pass_recall_std_percent": 4.9,
    },
])

def to_uci_row(method, summary):
    metric_map = summary.set_index("metric")
    return {
        "method": method,
        "ber_mean_percent": metric_map.loc["balanced_error_rate", "mean"] * 100,
        "ber_std_percent": metric_map.loc["balanced_error_rate", "std"] * 100,
        "fail_recall_mean_percent": metric_map.loc["fail_recall", "mean"] * 100,
        "fail_recall_std_percent": metric_map.loc["fail_recall", "std"] * 100,
        "pass_recall_mean_percent": metric_map.loc["pass_recall", "mean"] * 100,
        "pass_recall_std_percent": metric_map.loc["pass_recall", "std"] * 100,
    }

reproduction_comparison = pd.concat([
    uci_published,
    pd.DataFrame([
        to_uci_row("Approx F-test reproduction", f_test_summary),
        to_uci_row("Approx Pearson reproduction", pearson_summary),
    ]),
], ignore_index=True)

TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
uci_comparison_path = TABLE_DIR / "uci_reproduction_comparison.csv"
reproduction_comparison.to_csv(uci_comparison_path, index=False)

print(f"Saved UCI reproduction comparison: {uci_comparison_path.relative_to(PROJECT_ROOT)}")
reproduction_comparison.round(2)

Saved UCI reproduction comparison: reports\tables\uci_reproduction_comparison.csv


,method,ber_mean_percent,ber_std_percent,fail_recall_mean_percent,fail_recall_std_percent,pass_recall_mean_percent,pass_recall_std_percent
0,UCI F-test baseline,33.50,2.20,59.1,4.80,73.80,1.80
1,UCI Pearson baseline,34.10,2.00,57.4,4.30,74.40,4.90
2,Approx F-test reproduction,49.41,2.08,2.0,4.22,99.18,0.71
3,Approx Pearson reproduction,49.41,2.08,2.0,4.22,99.18,0.71


## 7. Discussion

This notebook should be read as an approximation, not an exact reproduction of the original UCI baseline. It is a benchmark-reproduction exercise, not the final project model. The main project results and final selected model are summarized in `notebooks/01_secom_fault_detection.ipynb`.

The fixed RBF Kernel Ridge configuration used here does **not** reproduce the published UCI F-test result. The reproduction attempt gives a much higher BER and very low Fail recall, meaning this particular implementation is too conservative and predicts almost all samples as Pass.

Possible reasons for differences include:

- the UCI notes do not specify the exact missing-value strategy;
- the kernel type and hyperparameters are not fully specified;
- the original classifier may have used a different thresholding rule or regularization setting;
- fold construction may differ;
- sklearn's implementation may differ from the original software implementation;
- small implementation details matter because the Fail class contains only `104` examples.

The important reproducibility guardrail is preserved: high-missingness removal, imputation, constant-feature removal, scaling, and feature selection are all inside the pipeline and are refit independently in every cross-validation fold.

## 8. References

Primary source for this notebook:

- McCann, M. & Johnston, A. (2008). **SECOM**. UCI Machine Learning Repository. DOI: [10.24432/C54305](https://doi.org/10.24432/C54305). Dataset page: [UCI SECOM](https://archive.ics.uci.edu/dataset/179/secom).

Implementation references:

- scikit-learn [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) and [common pitfalls / data leakage](https://scikit-learn.org/stable/common_pitfalls.html): used to keep preprocessing and feature selection inside cross-validation folds.
- scikit-learn [`SelectKBest`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html): used for top-`k` feature selection.
- scikit-learn [`f_classif`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.f_classif.html): used for the ANOVA F-test ranking approximation.
- scikit-learn [`KernelRidge`](https://scikit-learn.org/stable/modules/generated/sklearn.kernel_ridge.KernelRidge.html): used as the kernel ridge estimator behind the small classifier wrapper.
- scikit-learn [`balanced_accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.balanced_accuracy_score.html): used to compute `BER = 1 - balanced_accuracy` for comparison with the UCI balanced-error-rate table.
